In [2]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T

# 1. Configuración de la sesión (El Cerebro)
spark = SparkSession.builder \
    .appName("Analisis_Rapido") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

# 2. Definición de Esquema (Evita NameError)
schema = T.StructType([
    T.StructField("id", T.IntegerType(), True),
    T.StructField("fecha", T.TimestampType(), True),
    T.StructField("valor", T.DoubleType(), True),
    T.StructField("categoria", T.StringType(), True)
])

# 3. Ingesta (Lazy Load)
#df = spark.read.csv("archivo.csv", header=True, schema=schema)

# --- ESPACIO PARA TRANSFORMACIONES ---

# 4. Acción Final (Cómputo real)
# df.show(10) 
# df.write.mode("overwrite").parquet("resultado.parquet")

2. Guía de Manipulación (Las 7 operaciones fundamentales)

Estas son las transformaciones que usarás el 90% del tiempo:

    Selección y Alias: df.select(F.col("valor").alias("precio"), "categoria")

    Filtrado: df.filter((F.col("valor") > 100) & (F.col("categoria") == 'A'))

    Crear/Modificar Columnas: df.withColumn("iva", F.col("valor") * 0.16)

    Agregaciones: df.groupBy("categoria").agg(F.avg("valor").alias("media"), F.count("*").alias("total"))

    Ordenamiento: df.orderBy(F.col("valor").desc())

    Manejo de Nulos: df.fillna(0, subset=["valor"]) o df.dropna()

    Uniones (Joins): df1.join(df2, on="id", how="left")

# Registrar para usar SQL
df.createOrReplaceTempView("tabla_sensores")

# Sentencias más comunes:
query = """
SELECT 
    categoria, 
    AVG(valor) as promedio,
    COUNT(*) as registros
FROM tabla_sensores
WHERE valor IS NOT NULL
GROUP BY categoria
HAVING promedio > 50
ORDER BY promedio DESC
"""

# Ejecutar la consulta (devuelve un DataFrame)
df_sql = spark.sql(query)

df_sql.show()

In [5]:
#UDFs Pandas
"""
import pandas as pd
import numpy as np

@F.pandas_udf(T.DoubleType())
def normalizar_z(v: pd.Series) -> pd.Series:
    return (v - v.mean()) / v.std()

# Uso directo en el pipeline
df_stats = df.withColumn("z_score", normalizar_z(F.col("valor")))
"""

'\nimport pandas as pd\nimport numpy as np\n\n@F.pandas_udf(T.DoubleType())\ndef normalizar_z(v: pd.Series) -> pd.Series:\n    return (v - v.mean()) / v.std()\n\n# Uso directo en el pipeline\ndf_stats = df.withColumn("z_score", normalizar_z(F.col("valor")))\n'

In [8]:
"""
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.sql.window import Window

# Configuración inicial
spark = SparkSession.builder \
    .appName("AML_Detection_BBVA") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

# Esquema de transacciones bancarias
schema = T.StructType([
    T.StructField("customer_id", T.StringType(), False),
    T.StructField("txn_date", T.TimestampType(), False),
    T.StructField("amount", T.DoubleType(), False),
    T.StructField("txn_type", T.StringType(), True)
])

# Carga (Simulada)
#df_txns = spark.read.csv("transacciones_bancarias.csv", header=True, schema=schema)

# --- LAS WINDOW FUNCTIONS (EL CORAZÓN DEL AML) ---

# 1. Definimos la Ventana: Particionamos por CLIENTE y ordenamos por FECHA
# Usamos 'rangeBetween' para ventanas de tiempo (ej. 3 días en segundos: 3*24*60*60)
# O 'rowsBetween' para N transacciones previas.
window_spec = Window.partitionBy("customer_id").orderBy(F.col("txn_date").cast("long"))

# 2. Aplicamos métricas de comportamiento temporal
df_aml_features = df_txns \
    .withColumn("suma_acumulada_24h", 
                F.sum("amount").over(window_spec.rangeBetween(-86400, 0))) \
    .withColumn("promedio_movil_3d", 
                F.avg("amount").over(window_spec.rangeBetween(-259200, 0))) \
    .withColumn("txn_previa", 
                F.lag("amount", 1).over(window_spec)) \
    .withColumn("dif_tiempo_segundos", 
                F.col("txn_date").cast("long") - F.lag(F.col("txn_date").cast("long"), 1).over(window_spec))

# 3. Lógica de Negocio: Flag de Alerta
# Si la suma en 24h > 10,000 USD (Umbral regulatorio común)
df_alerts = df_aml_features.withColumn(
    "is_alert", 
    F.when((F.col("suma_acumulada_24h") > 10000) & (F.col("amount") < 2000), True).otherwise(False)
)

df_alerts.select("customer_id", "txn_date", "amount", "suma_acumulada_24h", "is_alert").show()
"""

'\nfrom pyspark.sql import SparkSession\nimport pyspark.sql.functions as F\nimport pyspark.sql.types as T\nfrom pyspark.sql.window import Window\n\n# Configuración inicial\nspark = SparkSession.builder     .appName("AML_Detection_BBVA")     .config("spark.sql.execution.arrow.pyspark.enabled", "true")     .getOrCreate()\n\n# Esquema de transacciones bancarias\nschema = T.StructType([\n    T.StructField("customer_id", T.StringType(), False),\n    T.StructField("txn_date", T.TimestampType(), False),\n    T.StructField("amount", T.DoubleType(), False),\n    T.StructField("txn_type", T.StringType(), True)\n])\n\n# Carga (Simulada)\n#df_txns = spark.read.csv("transacciones_bancarias.csv", header=True, schema=schema)\n\n# --- LAS WINDOW FUNCTIONS (EL CORAZÓN DEL AML) ---\n\n# 1. Definimos la Ventana: Particionamos por CLIENTE y ordenamos por FECHA\n# Usamos \'rangeBetween\' para ventanas de tiempo (ej. 3 días en segundos: 3*24*60*60)\n# O \'rowsBetween\' para N transacciones previas.\nwindow_

In [9]:
"""
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer, StandardScaler
from pyspark.ml.classification import GBTClassifier # Gradient Boosted Trees (Potente para AML)

# 1. Preprocesamiento: Convertir categorías a números e indexar
indexer = StringIndexer(inputCol="txn_type", outputCol="type_index")

# 2. Vectorización: El paso más importante de Spark ML
# Agrupamos todas las variables en una sola columna 'features'
feature_cols = ["amount", "type_index", "suma_acumulada_24h", "dif_tiempo_segundos"]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")

# 3. Escalado (Opcional pero recomendado para modelos basados en distancia)
scaler = StandardScaler(inputCol="raw_features", outputCol="features")

# 4. El Modelo
gbt = GBTClassifier(labelCol="is_alert", featuresCol="features", maxIter=10)

# 5. Pipeline: Encadena todo para evitar Data Leakage
pipeline = Pipeline(stages=[indexer, assembler, scaler, gbt])

# Entrenar
model = pipeline.fit(train_df)
predictions = model.transform(test_df)
"""

'\nfrom pyspark.ml import Pipeline\nfrom pyspark.ml.feature import VectorAssembler, StringIndexer, StandardScaler\nfrom pyspark.ml.classification import GBTClassifier # Gradient Boosted Trees (Potente para AML)\n\n# 1. Preprocesamiento: Convertir categorías a números e indexar\nindexer = StringIndexer(inputCol="txn_type", outputCol="type_index")\n\n# 2. Vectorización: El paso más importante de Spark ML\n# Agrupamos todas las variables en una sola columna \'features\'\nfeature_cols = ["amount", "type_index", "suma_acumulada_24h", "dif_tiempo_segundos"]\nassembler = VectorAssembler(inputCols=feature_cols, outputCol="raw_features")\n\n# 3. Escalado (Opcional pero recomendado para modelos basados en distancia)\nscaler = StandardScaler(inputCol="raw_features", outputCol="features")\n\n# 4. El Modelo\ngbt = GBTClassifier(labelCol="is_alert", featuresCol="features", maxIter=10)\n\n# 5. Pipeline: Encadena todo para evitar Data Leakage\npipeline = Pipeline(stages=[indexer, assembler, scaler, gb

In [10]:
"""
import shap
import pandas as pd

# Nota: Para modelos de árbol (GBT, Random Forest), usamos TreeExplainer
# Extraemos el modelo del pipeline
trained_gbt_model = model.stages[-1]

@F.pandas_udf(T.ArrayType(T.FloatType()))
def calcular_shap_values(pdf_features: pd.Series) -> pd.Series:
    # 1. Convertimos la serie de vectores a una matriz de NumPy
    data = np.array(pdf_features.tolist())
    
    # 2. Creamos el explicador (esto sucede en cada Executor en paralelo)
    explainer = shap.TreeExplainer(trained_gbt_model)
    shap_values = explainer.shap_values(data)
    
    # Devolvemos los valores SHAP (en GBT es un array por fila)
    return pd.Series(shap_values.tolist())

# Aplicar al DataFrame
df_with_shap = predictions.withColumn("shap_impact", calcular_shap_values("features"))
"""
"""
-- Feature Engineering rápido en SQL
SELECT 
    customer_id,
    amount,
    -- Ratio de la transacción actual vs el promedio histórico
    amount / AVG(amount) OVER(PARTITION BY customer_id) as relative_amount,
    -- Tiempo desde la última transacción
    unix_timestamp(txn_date) - LAG(unix_timestamp(txn_date)) OVER(PARTITION BY customer_id ORDER BY txn_date) as time_delta
FROM raw_data
"""

'\n-- Feature Engineering rápido en SQL\nSELECT \n    customer_id,\n    amount,\n    -- Ratio de la transacción actual vs el promedio histórico\n    amount / AVG(amount) OVER(PARTITION BY customer_id) as relative_amount,\n    -- Tiempo desde la última transacción\n    unix_timestamp(txn_date) - LAG(unix_timestamp(txn_date)) OVER(PARTITION BY customer_id ORDER BY txn_date) as time_delta\nFROM raw_data\n'

In [1]:
"""
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
import pyspark.sql.types as T
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer
# Nota: En entornos estándar se usa sparkxgb o el wrapper de XGBoost para PySpark
from xgboost.spark import SparkXGBClassifier 

# 1. Preparación del Entorno
spark = SparkSession.builder \
    .appName("BBVA_AML_XGBoost") \
    .config("spark.sql.execution.arrow.pyspark.enabled", "true") \
    .getOrCreate()

# 2. Ingeniería de Características (Features)
# Supongamos que ya vienen de nuestras Window Functions anteriores
input_cols = ["amount", "suma_acumulada_24h", "avg_txn_3d", "time_delta_secs"]

# Indexamos el tipo de transacción (Transferencia, Retiro, etc.)
indexer = StringIndexer(inputCol="txn_type", outputCol="type_idx", handleInvalid="keep")

# Ensamblamos el vector de características
assembler = VectorAssembler(inputCols=input_cols + ["type_idx"], outputCol="raw_features")

# Escalado (XGBoost es robusto, pero el escalado ayuda a la convergencia)
scaler = StandardScaler(inputCol="raw_features", outputCol="features")

# 3. Configuración del Modelo XGBoost
# Como Manager, destaca estos hiperparámetros:
# scale_pos_weight: Vital para desbalanceo de clases (Fraude vs No Fraude)
xgb = SparkXGBClassifier(
    features_col="features",
    label_col="is_fraud",
    num_workers=3,             # Ajustado a tu homelab (NUCs/Beelink)
    max_depth=6,
    eta=0.1,
    scale_pos_weight=99,       # Si el fraude es el 1%, le damos peso 99
    missing=0.0                # Manejo nativo de nulos
)

# 4. Construcción del Pipeline
pipeline_aml = Pipeline(stages=[indexer, assembler, scaler, xgb])

# 5. Entrenamiento con Validación Cruzada (Estrategia Manager)
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import BinaryClassificationEvaluator

grid = ParamGridBuilder() \
    .addGrid(xgb.max_depth, [4, 6, 8]) \
    .build()

evaluator = BinaryClassificationEvaluator(
    labelCol="is_fraud", 
    rawPredictionCol="probability", 
    metricName="areaUnderPR" # En AML, Precision-Recall es mejor que el Área bajo la curva ROC
)

cv = CrossValidator(estimator=pipeline_aml,
                    estimatorParamMaps=grid,
                    evaluator=evaluator,
                    numFolds=3)

# fit_model = cv.fit(train_df)
"""

'\nfrom pyspark.sql import SparkSession\nimport pyspark.sql.functions as F\nimport pyspark.sql.types as T\nfrom pyspark.ml import Pipeline\nfrom pyspark.ml.feature import VectorAssembler, StandardScaler, StringIndexer\n# Nota: En entornos estándar se usa sparkxgb o el wrapper de XGBoost para PySpark\nfrom xgboost.spark import SparkXGBClassifier \n\n# 1. Preparación del Entorno\nspark = SparkSession.builder     .appName("BBVA_AML_XGBoost")     .config("spark.sql.execution.arrow.pyspark.enabled", "true")     .getOrCreate()\n\n# 2. Ingeniería de Características (Features)\n# Supongamos que ya vienen de nuestras Window Functions anteriores\ninput_cols = ["amount", "suma_acumulada_24h", "avg_txn_3d", "time_delta_secs"]\n\n# Indexamos el tipo de transacción (Transferencia, Retiro, etc.)\nindexer = StringIndexer(inputCol="txn_type", outputCol="type_idx", handleInvalid="keep")\n\n# Ensamblamos el vector de características\nassembler = VectorAssembler(inputCols=input_cols + ["type_idx"], output

In [ ]:
"""
#Duplicados
# 1. Duplicados exactos (toda la fila es idéntica)
df_unique = df.distinct()

# 2. Duplicados basados en llaves (Ej: mismo ID de transacción)
# Es más común en ingeniería de datos
df_dedup = df.dropDuplicates(["transaction_id", "customer_id"])
"""


"""
#Nulos

# Borrar si TODA la fila es nula
df_clean = df.na.drop(how="all")

# Borrar si hay AL MENOS un nulo en las columnas críticas de AML
df_critical = df.na.drop(subset=["amount", "customer_id"])

# Borrar si hay más de 2 nulos en la fila (umbral de tolerancia)
df_threshold = df.na.drop(thresh=2)
"""

"""
#Imputación estadística

from pyspark.ml.feature import Imputer

# Definimos el imputador
imputer = Imputer(
    inputCols=["amount", "voltaje"], 
    outputCols=["amount_imputed", "voltaje_imputed"]
).setStrategy("median") # La mediana es más robusta a outliers en AML

# Entrenamos (calcula la mediana) y transformamos
model_imputer = imputer.fit(df)
df_imputed = model_imputer.transform(df)
"""

"""
# imputación condicional
df_smart_fill = df.withColumn(
    "currency", 
    F.when(F.col("currency").isNull() & (F.col("country") == "MX"), "MXN")
     .otherwise(F.col("currency"))
)
"""